# שיעור 01 - הקדמה לסוכני AI

ברוכים הבאים לשיעור הראשון בקורס **סוכני AI למתחילים**!

**סוכן AI** הוא תוכנית שמשתמשת בדגם שפה גדול (LLM) כמנוע ההסקה שלה ויכולה לנקוט *פעולות* בעולם האמיתי — לקרוא ל-APIs, לשאול מסדי נתונים, או להריץ קוד — כדי להשיג מטרה עבור המשתמש.

במחברת זו תבנו את הסוכן הראשון שלכם: **סוכן נסיעות** שממליץ על יעדי חופשה. בדרך תלמדו כיצד:

1. להתחבר לשירות Azure AI Foundry Agent באמצעות **מסגרת הסוכנים של מיקרוסופט**.
2. לתת לסוכן **כלי** — פונקציית פייתון פשוטה שהוא יכול לקרוא לה.
3. להריץ את הסוכן ולבדוק את התגובה שלו.
4. להזרים את תגובת הסוכן טוקן אחר טוקן.


## התקנה

לפני הרצת המחברת הזו, ודא שיש לך:

1. **פרויקט Azure AI Foundry** עם מודל צ'אט מופעל (למשל `gpt-4o-mini`).
2. **כניסה עם Azure CLI** — הרץ `az login` בטרמינל שלך.
3. **קביעת משתני הסביבה הנדרשים:**
   - `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Azure AI Foundry שלך.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל שהפעלת.

התא התחתון מתקין את החבילות פייתון הדרושות לך.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## יצירת הסוכן הראשון שלך

סוכן צריך שתי דברים:

- **הוראות** שמספרות לו *מי הוא* ו*איך להתנהג* (פירומט מערכת).
- **כלים** — פונקציות פייתון המעוטרות ב-`@tool` שהסוכן יכול לקרוא להן כדי לקבל מידע או לבצע פעולות.

מטה מוגדר כלי פשוט שמחזיר רשימה של יעדי חופשה פופולריים. הסוכן ישתמש בכלי זה כאשר משתמש יבקש המלצות לטיולים.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## תגובות בשידור חי

לחווייה אינטראקטיבית יותר, ניתן **לשדר** את תגובת הסוכן. במקום להמתין לתשובה המלאה, הסוכן מספק קטעי טקסט בזמן שהם נוצרים. זה שימושי במיוחד בממשקי צ'אט בהם רוצים להציג פלט בזמן אמת.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## סיכום

במסמך זה למדת כיצד:

- **ליצור ספק** שמתחבר לשירות Azure AI Foundry Agent באמצעות `AzureAIProjectAgentProvider`.
- **להגדיר כלי** באמצעות הדקורטור `@tool` כך שהסוכן יוכל לקרוא לפונקציות הפייתון שלך.
- **להפעיל את הסוכן** עם הודעת משתמש ולהדפיס את התגובה שלו.
- **לשדר תגובות** עבור פלט בזמן אמת.

בשיעור הבא נחקור את מסגרות הסוכנים לעומק רב יותר ונלמד כיצד לתת לסוכנים כלים חזקים יותר ויכולת חשיבה מרובת שלבים.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לזכור כי תרגומים אוטומטיים עלולים לכלול טעויות או אי דיוקים. המסמך המקורי בשפתו המקורית יש להחשיבו כמקור הרשמי והמהימן. למידע קריטי מומלץ להשתמש בתרגום מקצועי אנושי. אנו לא אחראים לכל אי הבנה או פרשנות שגויה הנובעות משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
